# ingest_api\nGenerated from 01_ingestion/ingest_api.py for Databricks notebook import.\n

In [ ]:
"""Generic API ingestion adapter."""\n\nfrom __future__ import annotations\n\nimport os\nimport time\n\nimport requests\n\n# Retry on transient server/rate-limit errors.\n_RETRY_STATUS_CODES = {429, 500, 502, 503, 504}\n\n# Common envelope keys different APIs use to wrap record arrays.\n# Checked in order; first match wins.\n_DATA_ENVELOPE_KEYS = ["data", "items", "results", "records", "value", "content"]\n\n\ndef _resolve_api_url(api_profile: dict, source: dict) -> str:\n    endpoint = source.get("api_endpoint", "")\n    base_url = api_profile.get("base_url", "")\n    if not endpoint:\n        raise ValueError("api_endpoint is required in source metadata")\n    if endpoint.startswith("http://") or endpoint.startswith("https://"):\n        return endpoint\n    if not base_url:\n        raise ValueError("base_url is required in API profile for relative api_endpoint values")\n    return f"{base_url.rstrip('/')}/{endpoint.lstrip('/')}"\n\n\ndef _resolve_headers(api_profile: dict, headers: dict | None = None) -> dict:\n    """Build request headers including authentication.\n\n    Supported auth_type values in the API profile:\n    - ``bearer``  — reads token from the env var named by ``token_env``\n    - ``api_key`` — reads key from env var ``api_key_env``; header name from\n                    ``api_key_header`` (default: ``X-API-Key``)\n    - ``basic``   — reads username from ``user_env`` and password from ``password_env``\n    - ``none``    — no auth header added\n    """\n    final_headers = dict(headers or {})\n    auth_type = str(api_profile.get("auth_type", "none")).lower()\n\n    if auth_type == "bearer":\n        token_env = api_profile.get("token_env", "")\n        token = os.getenv(token_env, "") if token_env else ""\n        if token:\n            final_headers.setdefault("Authorization", f"Bearer {token}")\n\n    elif auth_type == "api_key":\n        key_env = api_profile.get("api_key_env", "")\n        key_header = api_profile.get("api_key_header", "X-API-Key")\n        api_key = os.getenv(key_env, "") if key_env else ""\n        if api_key:\n            final_headers.setdefault(key_header, api_key)\n\n    elif auth_type == "basic":\n        import base64\n        user_env = api_profile.get("user_env", "")\n        password_env = api_profile.get("password_env", "")\n        user = os.getenv(user_env, "") if user_env else ""\n        password = os.getenv(password_env, "") if password_env else ""\n        if user and password:\n            token = base64.b64encode(f"{user}:{password}".encode()).decode()\n            final_headers.setdefault("Authorization", f"Basic {token}")\n\n    return final_headers\n\n\ndef _extract_records(payload: object, source: dict) -> list[dict]:\n    """Extract a flat list of records from the API response payload.\n\n    Supports:\n    - Plain list response: ``[{...}, {...}]``\n    - Enveloped response using a configurable or auto-detected key:\n      ``{"data": [...]}`` / ``{"items": [...]}`` / ``{"results": [...]}`` etc.\n    - Single-record response: ``{...}`` → wrapped in a list.\n\n    Set ``response_data_key`` in ``source_options_json`` to pin the envelope\n    key for a specific source (e.g. ``"response_data_key": "content"``).\n    """\n    if isinstance(payload, list):\n        return payload  # type: ignore[return-value]\n\n    if isinstance(payload, dict):\n        # Allow config-level override first.\n        override_key = source.get("response_data_key", "")\n        if override_key and isinstance(payload.get(override_key), list):\n            return payload[override_key]  # type: ignore[return-value]\n        # Auto-detect common envelope keys.\n        for key in _DATA_ENVELOPE_KEYS:\n            if isinstance(payload.get(key), list):\n                return payload[key]  # type: ignore[return-value]\n        # Scalar envelope — wrap the whole dict.\n        return [payload]\n\n    return []\n\n\ndef _parse_response_json(response: requests.Response) -> object:\n    """Parse response body as JSON with a clear error if content-type is wrong."""\n    content_type = response.headers.get("Content-Type", "")\n    if "json" not in content_type and "javascript" not in content_type:\n        snippet = response.text[:200]\n        raise ValueError(\n            f"Expected JSON response but got Content-Type: {content_type!r}. "\n            f"Response snippet: {snippet!r}"\n        )\n    return response.json()\n\n\ndef _fetch_page(\n    session: requests.Session,\n    url: str,\n    headers: dict,\n    params: dict,\n    timeout: int,\n) -> requests.Response:\n    response = session.get(url, headers=headers, params=params, timeout=timeout)\n    return response\n\n\ndef ingest_api(\n    source: dict,\n    api_profile: dict,\n    headers: dict | None = None,\n    params: dict | None = None,\n    source_options: dict | None = None,\n) -> list[dict]:\n    """Ingest records from a REST API endpoint with retry and pagination support.\n\n    Pagination is driven by ``source_options`` (parsed from ``source_options_json``):\n\n    .. code-block:: json\n\n        {\n          "pagination_type": "cursor",\n          "next_page_key":   "next_page_token",\n          "page_size_param": "limit",\n          "page_size":       500,\n          "max_pages":       50\n        }\n\n    ``pagination_type`` values supported:\n    - ``cursor``  — follows ``next_page_key`` in response body until null/absent\n    - ``offset``  — increments ``offset`` param by ``page_size`` until empty page\n    - ``none``    — single request (default)\n    """\n    if "timeout_seconds" not in api_profile:\n        raise ValueError("timeout_seconds must be configured in API profile")\n\n    opts = source_options or {}\n    endpoint = _resolve_api_url(api_profile, source)\n    timeout = int(api_profile["timeout_seconds"])\n    max_retries = int(api_profile.get("max_retries", 2))\n    backoff_seconds = float(api_profile.get("retry_backoff_seconds", 2))\n    final_headers = _resolve_headers(api_profile, headers=headers)\n\n    # All pagination config must come from source_options_json, not from the\n    # source registry row — they are operational options, not registry fields.\n    pagination_type = str(opts.get("pagination_type", "none")).lower()\n    next_page_key = str(opts.get("next_page_key", "next_page_token"))\n    page_size_param = str(opts.get("page_size_param", "limit"))\n    page_size = int(opts.get("page_size", 500))\n    max_pages = int(opts.get("max_pages", 100))\n\n    current_params = dict(params or {})\n    if pagination_type in {"cursor", "offset"} and page_size_param:\n        current_params[page_size_param] = page_size\n\n    all_records: list[dict] = []\n    offset = 0\n\n    with requests.Session() as session:\n        for page_num in range(max_pages):\n            if pagination_type == "offset":\n                current_params["offset"] = offset\n\n            # Retry loop for transient failures.\n            last_error: Exception | None = None\n            response: requests.Response | None = None\n            for attempt in range(max_retries + 1):\n                try:\n                    response = _fetch_page(session, endpoint, final_headers, current_params, timeout)\n                    if response.status_code in _RETRY_STATUS_CODES and attempt < max_retries:\n                        time.sleep(backoff_seconds * (attempt + 1))\n                        continue\n                    response.raise_for_status()\n                    break\n                except requests.RequestException as exc:\n                    last_error = exc\n                    if attempt >= max_retries:\n                        raise RuntimeError(\n                            f"API request failed after {max_retries + 1} attempt(s): {exc}"\n                        ) from exc\n                    time.sleep(backoff_seconds * (attempt + 1))\n\n            if response is None:\n                raise RuntimeError("API ingestion failed with unknown error")\n\n            payload = _parse_response_json(response)\n            records = _extract_records(payload, source)\n            all_records.extend(records)\n\n            # Determine whether to fetch the next page.\n            if pagination_type == "cursor":\n                next_token = payload.get(next_page_key) if isinstance(payload, dict) else None  # type: ignore[union-attr]\n                if not next_token:\n                    break\n                current_params[next_page_key] = next_token\n            elif pagination_type == "offset":\n                if not records:\n                    break  # Empty page → last page reached.\n                offset += len(records)\n            else:\n                break  # No pagination — single request.\n\n    return all_records\n